In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yfinance as yf

In [ ]:
ticker = "AAPL"
data = yf.download(ticker, start="2020-01-01", end="2024-01-01")

prices = data["Close"].dropna()

In [ ]:
log_returns = np.log(prices / prices.shift(1)).dropna()

In [ ]:
mu = log_returns.mean()
sigma = log_returns.std()

print("Mean:", mu)
print("Volatility:", sigma)

In [ ]:
n_simulations = 10000
n_days = 252
S0 = prices.iloc[-1].item()

returns = np.random.normal(mu, sigma, (n_days, n_simulations))
simulations = S0 * np.exp(np.cumsum(returns, axis=0))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(simulations[:, :50])
plt.title("Monte Carlo Simulated Price Paths (Normal)")
plt.xlabel("Days")
plt.ylabel("Price")
plt.show()

In [ ]:
normal_final = simulations[-1, :] / S0 - 1

In [ ]:
def var_es(returns, alpha=0.05):
    var = np.percentile(returns, alpha * 100)
    es = returns[returns <= var].mean()
    return var, es

var_n, es_n = var_es(normal_final)

print("Normal VaR:", var_n)
print("Normal ES:", es_n)

In [ ]:
df, loc, scale = stats.t.fit(log_returns)

print("df:", df, "loc:", loc, "scale:", scale)

In [ ]:
t_returns = stats.t.rvs(df, loc=loc, scale=scale, size=(n_days, n_simulations))
t_simulations = S0 * np.exp(np.cumsum(t_returns, axis=0))

In [ ]:
t_final = t_simulations[-1, :] / S0 - 1

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(normal_final, bins=50, alpha=0.5, label="Normal", density=True)
plt.hist(t_final, bins=50, alpha=0.5, label="Student-t", density=True)

plt.title("Return Distribution: Normal vs Student-t")
plt.legend()
plt.show()

In [ ]:
var_t, es_t = var_es(t_final)

print("\n--- Comparison ---")
print("Normal VaR:", var_n, "| ES:", es_n)
print("Student-t VaR:", var_t, "| ES:", es_t)

In [ ]:
x = np.linspace(log_returns.min(), log_returns.max(), 100)

plt.figure(figsize=(10,5))
plt.hist(log_returns, bins=50, density=True, alpha=0.6, label="Empirical")
plt.plot(x, stats.norm.pdf(x, mu, sigma), label="Normal Fit", linewidth=2)

plt.title("Empirical vs Normal Distribution")
plt.legend()
plt.show()

In [ ]:
rolling_vol = log_returns.rolling(20).std()

plt.figure(figsize=(10,5))
plt.plot(rolling_vol)
plt.title("Rolling Volatility (20-day)")
plt.xlabel("Time")
plt.ylabel("Volatility")
plt.show()

## Key Insights

- Financial returns exhibit fat tails, making the normal distribution a poor approximation for extreme events.
- Student-t simulations produce larger tail risk estimates (higher VaR and Expected Shortfall).
- Volatility is time-varying, violating the constant volatility assumption in standard Monte Carlo models.

This highlights the importance of model selection in risk estimation.